# Problem Solving and Debugging

> ### Learning Objectives
>
> By the end of this chapter you should be able to work with:
>
> - A structured, four-stage problem-solving model: Understand → Design → Implement → Evaluate
> - Analyzing a problem by asking what the desired outcome is, what is given, and what is missing (the "Double Diamond")
> - Designing a solution using pseudocode before writing Python
> - Recognizing when coding uncovers gaps in the design and being prepared to iterate
> - Designing test cases: normal cases, edge cases, and error cases
> - The three categories of errors: syntax, runtime, and semantic
> - Reading a stack trace to locate the source of a runtime error
> - Debugging by inspection and hand-tracing code
> - Debugging by printing variable values at key points
> - The Google Colab variable inspector
> - The Python debugger (`pdb`): `breakpoint()`, `c`, and `q`

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SETUP — run this cell first.
#
#  It draws every figure used in this chapter and switches the notebook into
#  "show me every result" mode.  Everything it needs is right here: nothing to
#  install, nothing to download, no other files required.
#
#  (Curious what a figure is made of?  The drawing code is all below.)
# ══════════════════════════════════════════════════════════════════════
import io

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Circle
from matplotlib.lines import Line2D
from IPython.display import Image, display
from IPython.core.interactiveshell import InteractiveShell

# echo the value of *every* expression in a cell, the way the Python prompt
# does — many examples in this book show several results at once
InteractiveShell.ast_node_interactivity = "all"

# ------------------------------------------------------------- drawing ---

INK = "#1a1a1a"

MUTED = "#6b7280"

FILL = "#eef2f7"

ACCENT = "#2563eb"

WARM = "#b45309"

EDGE = "#334155"

def _frame(ax, xlim, ylim, title=None):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10, color=MUTED, pad=8)

def _box(ax, xy, text, w=2.6, h=0.9, fc=FILL, ec=EDGE, fs=9, bold=False):
    x, y = xy
    ax.add_patch(FancyBboxPatch(
        (x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK,
            zorder=3, fontweight="bold" if bold else "normal")
    return xy

def _diamond(ax, xy, text, w=3.0, h=1.5, fc="#fff7ed", ec=WARM, fs=9):
    x, y = xy
    ax.add_patch(Polygon(
        [(x, y + h / 2), (x + w / 2, y), (x, y - h / 2), (x - w / 2, y)],
        closed=True, linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK, zorder=3)
    return xy

def _dot(ax, xy, r=0.09):
    ax.add_patch(Circle(xy, r, facecolor=EDGE, edgecolor=EDGE, zorder=4))
    return xy

def _arrow(ax, pts, label=None, label_at=0.5, label_off=(0.0, 0.18),
           color=EDGE, ha="center"):
    """Poly-line arrow through `pts` (elbow routing), head on the last segment."""
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    ax.add_line(Line2D(xs[:-1] + [xs[-1]], ys[:-1] + [ys[-1]],
                       color=color, linewidth=1.3, zorder=1,
                       solid_capstyle="round"))
    ax.annotate("", xy=pts[-1], xytext=pts[-2],
                arrowprops=dict(arrowstyle="-|>", color=color, linewidth=1.3,
                                shrinkA=0, shrinkB=0), zorder=1)
    if label:
        i = max(0, min(len(pts) - 2, int(label_at * (len(pts) - 1))))
        mx = (pts[i][0] + pts[i + 1][0]) / 2 + label_off[0]
        my = (pts[i][1] + pts[i + 1][1]) / 2 + label_off[1]
        ax.text(mx, my, label, fontsize=8, color=MUTED, ha=ha, va="center")

def _line(ax, pts, color=EDGE):
    """Poly-line with no arrowhead — for merging branches into a shared rail."""
    ax.add_line(Line2D([p[0] for p in pts], [p[1] for p in pts], color=color,
                       linewidth=1.3, zorder=1, solid_capstyle="round"))

def _cellgrid(ax, values, origin=(0, 0), cw=1.0, ch=1.0, fs=13, fc="white"):
    """A row/table of boxed cells; `values` is a list of rows."""
    x0, y0 = origin
    for r, row in enumerate(values):
        for c, v in enumerate(row):
            x = x0 + c * cw
            y = y0 - r * ch
            ax.add_patch(plt.Rectangle((x, y - ch), cw, ch, facecolor=fc,
                                       edgecolor=EDGE, linewidth=1.2, zorder=2))
            ax.text(x + cw / 2, y - ch / 2, str(v), ha="center", va="center",
                    fontsize=fs, color=INK, family="monospace", zorder=3)

def _mockwindow(ax, w, h, title, body, titlebar="#d7dde5", face="#ffffff",
                fs=9, textcolor=INK):
    """A framed window with a title bar and monospaced body lines."""
    ax.add_patch(plt.Rectangle((0, 0), w, h, facecolor=face, edgecolor=EDGE,
                               linewidth=1.2, zorder=1))
    ax.add_patch(plt.Rectangle((0, h - 0.55), w, 0.55, facecolor=titlebar,
                               edgecolor=EDGE, linewidth=1.2, zorder=2))
    ax.text(0.2, h - 0.28, title, fontsize=9, va="center", color=INK, zorder=3)
    y = h - 1.05
    for line, colour in body:
        ax.text(0.25, y, line, fontsize=fs, va="center", family="monospace",
                color=colour or textcolor, zorder=3)
        y -= 0.5

def _index_grid(ax, items, top_label, side_label, fs=13):
    n = len(items)
    _cellgrid(ax, [items], origin=(0, 1), fs=fs)
    for i in range(n):
        ax.text(i + 0.5, 1.25, str(i), ha="center", va="bottom", fontsize=10,
                color=ACCENT)
    if top_label:
        ax.text(-0.25, 1.3, top_label, ha="right", va="bottom", fontsize=9,
                color=ACCENT)
    if side_label:
        ax.text(-0.25, 0.5, side_label, ha="right", va="center", fontsize=9,
                color=ACCENT)
    _frame(ax, (-6.4, n + 0.4), (-0.4, 2.0))

def _double_diamond(ax, stage=None):
    names = ["Understand", "Design", "Implement", "Evaluate"]
    # two diamonds: centres at x=2.6 and x=7.8, half-width 2.6, half-height 2.0
    for d, cx in enumerate((2.6, 7.8)):
        left, right, top, bot = cx - 2.6, cx + 2.6, 2.0, -2.0
        for half in (0, 1):
            name = names[2 * d + half]
            tri = ([(left, 0), (cx, top), (cx, bot)] if half == 0
                   else [(cx, top), (right, 0), (cx, bot)])
            on = (name == stage)
            ax.add_patch(Polygon(tri, closed=True, zorder=1,
                                 facecolor="#b9bfc7" if on else "#eceef1",
                                 edgecolor="none"))
            tx = cx - 1.3 if half == 0 else cx + 1.3
            ax.text(tx, 0, name, ha="center", va="center", fontsize=10,
                    color=INK if on else MUTED,
                    fontweight="bold" if on else "normal", zorder=3)
        ax.add_patch(Polygon([(left, 0), (cx, top), (right, 0), (cx, bot)],
                             closed=True, facecolor="none", edgecolor=INK,
                             linewidth=2.2, zorder=2))
        ax.plot([cx, cx], [top, bot], color=INK, linewidth=1.0, zorder=2)
    for x, y in ((0.0, 0.0), (5.2, 0.0), (10.4, 0.0)):
        ax.add_patch(Circle((x, y), 0.22, facecolor="#c9ced6", edgecolor=INK,
                            linewidth=1.2, zorder=4))
    ax.plot([-1.5, -0.22], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([10.62, 11.9], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([5.2, 5.2], [-0.22, -3.1], color=INK, linewidth=1.6, zorder=2)
    ax.text(-1.7, 0, "Problem", ha="right", va="center", fontsize=10)
    ax.text(12.1, 0, "Program", ha="left", va="center", fontsize=10)
    ax.text(5.2, -3.35, "Specification", ha="center", va="top", fontsize=10)
    _frame(ax, (-4.2, 14.2), (-4.2, 2.6))

def draw_double_diamond(ax):
    """Replaces Images/doublediamond.png."""
    _double_diamond(ax)

def draw_double_diamond_understand(ax):
    """Replaces Images/doublediamond-understand.png."""
    _double_diamond(ax, "Understand")

def draw_double_diamond_design(ax):
    """Replaces Images/doublediamond-design.png."""
    _double_diamond(ax, "Design")

def draw_double_diamond_implement(ax):
    """Replaces Images/doublediamond-implement.png."""
    _double_diamond(ax, "Implement")

def draw_double_diamond_evaluate(ax):
    """Replaces Images/doublediamond-evaluate.png."""
    _double_diamond(ax, "Evaluate")

def draw_flow_test_debug(ax):
    """Replaces the TikZ flowchart in 15_Testing_and_Debugging
    (also published as testdebugprocess.pdf, which a notebook cannot show)."""
    _box(ax, (0, 8.4), "code", w=2.4)
    _diamond(ax, (0, 6.8), "update tests?", w=3.2)
    _box(ax, (0, 5.2), "test case generation / update", w=5.0)
    _box(ax, (0, 3.8), "write / update test driver", w=5.0)
    _box(ax, (0, 2.4), "run test driver", w=5.0)
    _diamond(ax, (0, 0.7), "faults detected?", w=3.6)
    _box(ax, (4.8, 0.7), "debug", w=2.2)
    ax.text(0, -1.1, "done (for now!)", ha="center", va="center", fontsize=9,
            color=INK)
    _arrow(ax, [(0, 7.95), (0, 7.55)])
    _arrow(ax, [(0, 6.05), (0, 5.65)], "yes", label_off=(-0.35, 0), ha="right")
    _arrow(ax, [(0, 4.75), (0, 4.25)])
    _arrow(ax, [(0, 3.35), (0, 2.85)])
    _arrow(ax, [(0, 1.95), (0, 1.45)])
    _arrow(ax, [(1.8, 0.7), (3.7, 0.7)], "yes", label_off=(0, 0.24))
    _arrow(ax, [(4.8, 1.15), (4.8, 8.4), (1.2, 8.4)])
    _arrow(ax, [(0, -0.05), (0, -0.85)], "no", label_off=(-0.32, 0), ha="right")
    _arrow(ax, [(-1.8, 6.8), (-3.6, 6.8), (-3.6, 2.4), (-2.5, 2.4)], "no",
           label_off=(0, 0.24))
    _frame(ax, (-5.4, 6.6), (-1.7, 9.2))

# ---------------------------------------------------------------- runtime ---
_SIZES = {'double_diamond': (8.2, 4.2), 'double_diamond_understand': (8.2, 4.2), 'double_diamond_design': (8.2, 4.2), 'double_diamond_implement': (8.2, 4.2), 'double_diamond_evaluate': (8.2, 4.2), 'flow_test_debug': (6.6, 7.4)}
_WIDTHS = {'double_diamond': 700, 'double_diamond_understand': 700, 'double_diamond_design': 700, 'double_diamond_implement': 700, 'double_diamond_evaluate': 700, 'flow_test_debug': 520}
_FIGURES = {}


def _render(name):
    fig, ax = plt.subplots(figsize=_SIZES.get(name, (6.4, 4.4)), dpi=110)
    globals()["draw_" + name](ax)
    fig.tight_layout(pad=0.3)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return buf.getvalue()


def show(name, width=None):
    """Display one of this chapter's figures."""
    display(Image(_FIGURES[name], width=width or _WIDTHS.get(name, 560)))


for _n in ['double_diamond', 'double_diamond_understand', 'double_diamond_design', 'double_diamond_implement', 'double_diamond_evaluate', 'flow_test_debug']:
    _FIGURES[_n] = _render(_n)

print("Setup complete \u2014 6 figure(s) ready.")


### Problem Solving Processes

In the previous chapters we have exposed you to a number of skills that are useful for trying to create new solutions to problems.  We have presented the tools of computational thinking, discussed how data is stored in variables, provided details of how algorithms are defined and specified, and introduced a number of key programming constructs that you find in many languages.  If we were in a biology or chemistry classroom, we might think of these as the pipettes and beakers of computer science.

In this chapter we are going to present the process of problem solving.  We are going to take a step back from our detailed look at different aspects of code and return to thinking about computational thinking and the art that is problem solving.  You are now armed with many of the basics types of sequential programming tools that are needed to solve lots of different problems, and in this chapter we will explore the process of moving from a problem to code.

#### Where do I start?

The question of "Where do I start?" is one that students in computer science have asked basically since we began doing computational thinking.  When faced with a problem, how do you get started?

A lot of students will jump in and start with variable names, and considering what they might hold, and might even tinker with a bit of code to see if they can get some input/output working.  The thing is though, that often leads to students being in the state of thrashing back and forth in their solution, trying to both problem solve and code at the same time, and this leads to frustration, anger fear and ultimately to people walking away from computer science.

In this course we are adapting a method that has been used since the 1950s in mathematics called *How to solve it* (Most recent edition: G. Polya, How to Solve It: A New Aspect of Mathematical Method, Oct. 2014, Princeton University Press), and has been adapted successfully in several other places since the 1990s in helping students learn and love problem solving in computer science.

One of the key features of problem solving in computational thinking is that it is a process that has two distinct phases.   The first phase of the process is about defining a *specification* that can be implemented and tested in the second phase.  This process can be visualized as two diamonds that meet at a single point, the specification, as in Figure~.

In [ ]:
show("double_diamond")

*Figure: The double diamond model of problem solving in programming.*

In the first phase you explore the problem to *understand*  the problem more fully, challenging the initial description to refine it into something that can be solved with an initial *specification* .  This results in a design space where there are a number of possible solutions that could solve that now well understood problem.  After we have a sufficient understanding of the problem, we can *design*  a solution by examining the problem from different angles, comparing it to other solutions, and translating the natural language into an intermediate language such as pseudocode or flowcharts to refine our specification into something that can be *implemented*.

Once we have a specification as to what the program should do, we can move into the second phase where we begin to write programming code. This solution space requires us to challenge our own assumptions in the specification and identify how we can write concise programming code that can solve the problem.  Finally, when we have a solution, we need to evaluate  that solution along a number of different axes to identify whether or not we have successfully solved the problem.

#### Working Example

It is always better to work with a concrete example.  Given that we live on a small island in the maritime region of Canada on the north Atlantic ocean, let's say that we want to write a program with the following specification:

> **Initial Problem Statement**
>
> Your program should find the highest year of snowfall on Prince Edward Island for any given period of time in years as specified by a user.

At first glance, this seems like a pretty simple problem.  However, when you begin to pull the problem apart, you will find there are a number of key things to think about.

But where do you start?

##### Understanding

The first step in solving a problem such as this one is the need to *understand* the problem thoroughly before taking on the challenge of designing a new solution.  Now, this may sound kind of silly --- after all, you just read that and understood it pretty well right?  Well, yes and no.  You understood the broad strokes, but part of problem solving at this stage is challenging that specification with a number of questions that you need to know if you are going to write a successful algorithmic solution.  These questions well help you as a programmer build up a mental model of what the solution to the problem is and how it is likely to look, and to interrogate what you know (certainty), what you know you don't know (epistemic uncertainty) and in some cases what is impossible to know, or the unknown unknowns (aleatoric uncertainty).

In [ ]:
show("double_diamond_understand")

*Figure: The problem solving begins with understanding the problem in detail to create the design space for the solution.*

There are a variety of questions that we can ask to challenge whether this problem is well specified:

1. What are the inputs to the program?
2. What are the outputs of the program?
3. Is there sufficient information to solve the problem?
4. Is there any redundant information that can be ignored?
5. Are there parts of the problem that are contradictory?
6. Are there parts of the problem that are ambiguous?
7. Can we break the problem into parts?

Looking at this problem description, we can see that the start year and the end year are likely to be our inputs.  So that seems to be straight forward.  However, there is insufficient information about those inputs --- for instance, what is the format of how those years will be input and stored?  While this seems unimportant for this problem, early in computing history a decision was made that computers would only represent years with 2 digits, assuming the first 2 to be 19.  This led to the panic in the last part of the twentieth century of Y2K (Author Power had his first programming job doing regression testing for Y2K in an insurance company.), where dates would roll over to being 1900 possibly sending systems into chaos, where concerted effort was made renovating systems to prevent that from happening (Unfortunately, one of the solutions only pushed the problem off to 2020 as a bunch of systems broke that year including, famously, the WWE 2020 video game.  If you are interested in what is coming next, C Programming has a year 2038 problem where good C programmers are set to make a fortune 2 decades from now).  So, we probably can make an assumption that we should store those years as a 4 digit number to avoid bad things from happening in the year 2100 when our clever program will no doubt still be being used!

What about the outputs?   Well, it is pretty ambiguous about what is supposed to happen.  The program says that it is supposed to find the highest year of snowfall on the island.  What is the result though?  Does the program report the year?  Does it report the snowfall?  Should it just return the value somewhere like a function, or does it need to print it for a user?  This very simple question has already revealed a ton of ambiguity.  We have a few options as to what we might do now to resolve these ambiguities.  The best solution would be to ask a *stakeholder* what the program is actually supposed to do.  A stakeholder is an individual who has some vested interest in seeing the creation of the program be successful.  In a university class such as this one, it is probably one of the instructors.  In the "real world" (The authors will endeavour to always put the words "real world" in quotes given that there is no idealized "real world"), there will be a number of different people from which you could seek clarification.  If you were a product manager working with clients, you might seek out the commissioner of the program and ask them.  If you were a software engineer, you might ask your project manager or software architect, or a more senior member of the team.  If you were a user experience designer, you might ask users directly what they would like.  If you are a data scientist, you might ask whoever will be consuming the data, such as external programmers, you would maybe ask them what data they need coming out of your program.  You might even ask all of these people what is supposed to happen in a complex enough problem.

Let's pretend that you went and asked the person who asked us to write the program, and they responded that it would be sufficient to have the program return an integer so that result can be used by a number of different programs. This results in the following refined problem statement:

> **Refined Problem Statement**
>
> *For a range of years, between a provided start year and end year*, your program should find for the highest year of snowfall on Prince Edward Island ~~ for any given period of time in years as specified by a user~~ *and return it in integer format*.

At this point we can ask a different questions that help derive an initial specification:

- Can we write a natural language description of the solution?
- Can we break the problem into parts?

If we were doing this by hand, we might ask the local archives for the snowfall for each previous year.  In most cases, snowfall will be recorded by day, so we then go through that data an entry at a time calculating the year with the deepest snowfall:

> **Natural Language Solution**
>
> Get the start and end year required, and then request the data for between those years.  Starting with the first year, add up all of the days of snowfall for one year, and record it as the highest snowfall.  For each subsequent year, add up all of the days of snowfall and compare that value to the highest previously recorded snowfall with the new value.  If that year has higher snowfall, then record it as such.  When we reach the end of the list, return the highest value as an integer.

We largely have the information that we need now to do a basic design, but before we go, we can see from the description that there are at least 5 parts to this problem:

- Get the range of years
- Get the data for the snowfall
- Calculate the snowfall for each year
- Compare the snowfall for each year to find the highest
- Return the highest snowfall

These different parts all have a variety of different ways that they can be solved.  In the design section of the process, we can take this very broad specification and hone in on one solution.

##### Design

Taking the basic specification that we now understand, we can begin to refine it in increasingly formal ways until we have a specification that can be implemented.

In [ ]:
show("double_diamond_design")

*Figure: Once the problem is understood, we can design a specific solution from the set of different possible solutions and produce a specification.*

Specifications can come in many forms.  In some cases they are in intermediary semi-formal languages such as pseudocode or flowcharts, like what we use in this class.  In some contexts, such as those where safety and security are involved, they may use more formal mechanisms to derive an exact specification.

Looking at our natural language specification we can now begin to challenge our specification to narrow down the solution:

- Have you solved a problem like this before?
- Has anyone else solved a problem like this before?
- Are there more general problems that are similar that we could solve?
- Are there more specific problems that are similar that we could solve?

By asking these questions we challenge ourselves regarding what the best ways to solve this problem might be.  If someone has already solve it, or better yet if you have done something similar, then you can go back and look at your previous solutions which could be instructive in how to solve this new problem.

Let's consider what you have already seen in this course.  Book-ending the solution, you know that we need 2 year values coming into our programmed solution, and you have to return the highest snowfall.  Well, this starts to point to our program being a function because it can handle both of those parts --- arguments and a return value.  The other parts of the problem are a bit more complicated.

In both calculating the snowfall and in comparing years we see the word "each".  This implies that we are going to have to do something repeatedly.  So we know that our final solution will have a loop of some kind in it that works at the level of years --- where we need to work through the entirety of the range of years supplied in the function.

> **How would you calculate snowfall?**
>
> Before we continue --- consider the problem of calculating the snowfall for a year.  Assuming you can have any data you like, what are the different ways you might do that?

When we look at calculating the snowfall, it entirely depends on what data we have available. Theoretically the data could come in any form, but given we understand how weather works from our everyday lives, we could probably assume for the moment that it could be any of the following:

- Snowfall per year
- Snowfall per month
- Snowfall per day
- Snowfall by hour
- Snowfall by minute

In the first case, you will only need a single loop.  For any of the others, you will need to add up a bunch of snowfall data, which is a more specific version of the problem of adding up a set of real numbers.  You know that this also can be done through a repetition --- which you encountered in Chapter~ as two loops nested inside of one another.

By decomposing the problem into parts, the problem can now able to refined, with one of those parts being much more detailed:

```

Refinement 1 HighestSnowfall

Get the start year and the end year.
For each year from the start year to the end year
     \textit{For each data point for the year}  
         \textit{Get the data for the data point} 
         \textit{Add the data point to a sum for the year} 
Compare the snowfall for each year to find the highest
Return the highest snowfall

```

Now, what about the problem of comparing the snowfalls over multiple years?  The most general solution would be to store a big list of all of the yearly values that grows based on the size of range, and then run through it looking for the biggest number.

At the moment, you do not know of any way to store sets of values in a list so you can do that comparison (see Chapter~) and so you need a different way to do it.

What if you used the more specific solution?  Instead of comparing all of the numbers, only compare two numbers?  You know how to do that --- is there a way to do that in this solution?  You could store the current highest number, and then compare each new number that is calculated in your loops to that known highest.  This means that instead of pre-calculating all of the years first and then comparing them, we will need to compare as we calculate a year's snowfall.  That will further refine the solution:

```

Refinement 2 HighestSnowfall

Get the start year and the end year.
For each year from the start year to the end year
    For each data point for the current year
        Get the data for the data point
         Add the data point to a sum for current year     
         \textit{Compare current year snowfall with known highest year}
         \textit{If the current year snowfall is higher, then}
             \textit{Current year is now the highest}
Return the highest snowfall

```

One question that is often asked by students is also: where do I stop?  You could continue to refine this, but here are a few heuristics to help you decide if you need more refinement:

- Did you use all of the inputs?
- Are there any special conditions you missed?
- Can you think of other new ways to solve the things you have already refined?
- Can you decompose any of your steps further into individual operations?
- Are you at the lowest level of abstraction where you have atomic data, or collections of atomic data?
- Do parts needing refinement require things that you do not control?

With the second refinement we are now at a pretty low level of abstraction.  You have a number of atomic pieces of data, the operations on that data, addition, numeric comparisons, and some repetitions and decisions that need to be executed.

The only remaining part of the refinement that we have not decomposed is the problem of getting the data.  You do not know where or how that will happen, so for the moment it will need to be left as is, until we get into writing the program.

Thus, the second refinement of this algorithm becomes our *specification*   moving into the programming stage.

##### Implement

After we have our specification, we can move from designing the program to actually writing the code in an implementation of the solution.

In [ ]:
show("double_diamond_implement")

*Figure: After refining a design to a specification, we can begin to implement the program.*

In this stage, you can identify a number of specific questions about how your code should be written:

- What are the variables that will be needed?
- What types do we expect the variables to be?
- What should the variables be initialized to?
- Are repetitions bounded, and thus for loops, or are they ended by a condition, and thus a while loop?
- Are there special conditions that have been missed at decision points?
- Where is the input to the program coming from?

As our specification is quite precise in the data that is needed.  You will need a variable to hold the snowfall for the current year being calculated, which could be called `current` and one for the highest encountered snowfall thus far in the program, which could be called `highest`.

> **How should the variables be initialized?**
>
> Before you read the solution in the next section, what do you think the variables should be initialized to, and where should that initialization happen?

From the previous design discussion, the inputs to the program will be provided through two arguments to a function.  However, you might challenge what should happen if the start year and end year are the same?  What behaviour would be expected of the program in that case?  What if the end year was lower than the start year?  These are all things that would need to be accounted for in your program.

Finally, we have the sticky problem of: where will the data come from for the snowfall?  As discussed, you do not have many programming tools for managing lists of numbers, so for now we are going to pretend that there is a function that will take the day, from 1 to 365, and the year, and return snowfall for us for that day.

This results in the following Python program:

> *A fragment for illustration — it is not complete enough to run.*
```python
def highest_snowfall(start, end):
    current = 0
    highest = 0
    if end < start:
        for year in range(start,end+1):
            for day in range(1,365):
                current = current + snowfall(day, year)
            if highest < current:
                highest = current
            current = 0
        return highest
    else:
        return -1
```

At the moment, you have only a limited set of tools at your disposal.  When we come towards the end of the course we will return to this specification and its related program and present some details of what else we might ask.

##### Evaluation

Evaluation of programs that we have developed can take many forms, depending on what you what to know about your program.

In [ ]:
show("double_diamond_evaluate")

*Figure: Once we have an implementation that is of sufficient fidelity that we can begin to evaluate it for various qualities.*

When we think of evaluation of programming, it most often comes in the for of fixing errors.  Whether it be syntax, logic or run-time errors, there will always be a need to debug and fix a program.  We will tackled this in more detail in a later chapter, but suffice to say that even in this simple program there is at least 1 bug that you can find by asking:

- Are there any conditions not currently considered?
- What happens if variables are set to unexpected values?
- Can you generate a variety of different edge cases?
- Does the program satisfy the requirements of what it was supposed to do?

In this case we might ask some questions about what will happen if the years come in as unexpected values, like negative values or 0.  You can also test for various very high numbers and very low numbers for years to see how things react.  You could also test with known values were we know the outcome.  For those who lived on PEI in the last two decades, you probably want to test 2015, also known as the "snowpocalypse" year.

All of these test will take your broad code base and refine it into a working automation of your specification.

Beyond testing for bugs, there are a number of other things that we may want to evaluate for in our code.  For example we might want to ask questions such as:

- Are there ways to organize data that will store it more compactly to take less space?
- Are there ways to specify an algorithm so that it will take fewer steps and thus save processing times?
- Are there ways to make the algorithm more general to handle other problems like it?
- If we were to start again, are there any other ways this problem might be solved?

The trade-off between using more space to save time, or being willing to sacrifice performance in time to save space, is a common discussion in how we build our programs.  For example, if we were trying to record the snowfall on an extra-terrestrial planet such as the moon Europa of Jupiter, we might have constraints both on the space with which to store data and constraints in battery life meaning we need as few processor steps as possible.

These types of discussions will continue throughout your computing career.  Time and space trade-offs will appear again in CS1920.  The notion of efficient organization of data will recur in 2910 and 2920.  However, it is never to early to challenge whether your code can be made more efficient.

#### Summary

In this chapter we have explored the process of problem solving.  While each problem is different, and will present it's own unique specifications and solutions, the questions contained herein can help you challenge your understanding, design and ultimately your programming code to help you build better programs in a structured way.

### What are Testing and Debugging?

Testing and debugging are processes to ensure that a program is correct in the sense that it behaves as expected, producing the correct outputs for given inputs.  If a program does not behave as expected, this is called a *fault* (also called an *error* or *bug*).

Testing is a **proactive**  process where one specifically chooses inputs or designs usage scenarios meant to determine if the program behaves correctly under those conditions.  For example, if we have a program that takes a floating-point value $r$ as input and is expected to output the area of a circle of radius $r$, then we might choose to test it by inputting a value of 4 and checking whether the output is correct (it should be about 50.27).  Or we might input a negative value for $r$ to test whether the program behaves reasonably when we give it unreasonable input.  In other words, we use testing to **detect** faults, hopefully before the software is released and faults are detected by customers in the course of normal usage!

Debugging is a **reactive**  process where, having detected a fault, we attempt to determine the reason for the fault and **repair** the fault.  For example, in our program that computes areas of circles, the expected output if a negative number is entered would be an error message indicating that the input radius needs to be positive.  But  if we input -4 and get an answer of -50.27 instead then our testing detected a fault --- the program didn't respond to the invalid input as expected!  We now have to determine why the fault occurred, and repair it.  Sometimes this can just be done by looking at the program and noticing where we made a mistake.  All too often, however, the reason for a fault occurring in a program is not obvious.  The larger and more complex a program is, the less obvious it becomes what the cause of a fault is likely to be.  We will look at various *debugging* techniques that can help us find and repair faults.

### Testing

The goal of the testing process is to detect all of the faults in some code.
To achieve this goal, we start by coming up with a set of *test cases*.  A test case consists of a specific input to the code, or a usage scenario performed under specific conditions.  When we test code, we want to generate a set of test cases that is:

1. as small as possible; and
2. has a very high likelihood of uncovering every fault that might exist in the code.

It is worth noting at this point that it is rare to test an entire program all at once with a single set of test cases.  More frequently, we generate test cases for an individual function we have written to make sure it is correct before moving on and writing other code that uses that function.  This is because trying to test an entire program all at once is quite unmanageable (the set of test cases becomes much too large) for all but the smallest programs.  If we test a program one function at a time, we can assume that previously written functions are correct when testing our most recently written function, which speeds test case generation.  That said, the process of testing is essentially the same whether we are testing just one small function of a larger program, or an entire program.

#### Standard Form of Test Cases

A test case is defined by determining the following items:

1. the input(s) for the test case;
2. the expected output(s) for the given input(s); and
3. the reason for the test case

When writing test cases, we will use the following standard form:

| Input | Expected output | Why this test case |
|---|---|---|
| `Description of program or function required inputs.` | `Description of expected program or function outputs.` | Description of reason for test case. |

This still leaves the question of how to actually identify test cases for a program or function.  There are two approaches we can use to generate test cases:  *white-box testing*, and *black-box testing*.

#### Test Case Generation: Black-Box Testing

If we generate test cases for an algorithm/program/function using knowledge of only the expected behaviour of the program or function, and without knowledge of the actual code, this is known as *black-box testing*.  The name is a metaphor:  you imagine the code is in an opaque black box.  You may test it by feeding input into the box, and receiving output from the box and checking whether it is correct, but you cannot see the code inside the box.   Test cases are generated by considering the different inputs that might be provided to the code including common, rare, unusual, and erroneous inputs.

Let's consider the following function:

In [ ]:
def is_divisible_by_7(numbers):
	"""
	This function returns true if the list numbers 
	contains a number that is divisible by 7, 
	and returns false otherwise.
	numbers: list of numbers to check
	return: if list contains a number divisible by 7
	"""
	for i in numbers:
		if i % 7 == 0:
			return True
			
	return False

As we have said, in black-box testing we are not supposed to look at the code for the algorithm when generating test cases.   So we must generate test cases using only the following knowledge:

> *A fragment for illustration — it is not complete enough to run.*
```python
def is_divisible_by_7(numbers) 
	"""
	This function returns true if the list numbers 
	contains a number that is divisible by 7, 
	and returns false otherwise.
	numbers: list of numbers to check
	return: if list contains a number divisible by 7
	"""
```

We now write test cases by considering typical, rare, and erroneous inputs and their expected outputs.  Below is a list of test cases we might come up with.  Though it is not normally necessary, we have tagged each test case with the label "common", "rare", or "erroneous" so you can better understand our thinking.

| Input | Expected output | Why this test case |
|---|---|---|
| `[1,2,7,3,5]` | `True` | Test when there is one element divisible by 7 in the middle of a list with many elements. [Common] |
| `[1,2,4,3,7]` | `True` | Test when there is one element divisible by 7 at the end of a list with many elements. [Rare] |
| `[14,2,4,3,6]` | `True` | Test when there is one element divisible by 7 at the beginning of a list with many elements. [Rare] |
| `[13,2,4,3,6]` | `False` | Test when there is no element divisible by 7 in a list with many elements. [Common] |
| `[7]` | `True` | Test when the list has only one element that is divisible by 7. [Rare] |
| `[9]` | `False` | Test when the list has only one element that is not divisible by 7. [Rare] |
| `[]` | `False` | Test when the list is empty [Rare] |
| `[2, 4, 'seventeen', 14]` | `False` | Test when the list contains something that is not a number. [Erroneous] |

Notice that none of the test cases rely on knowing the implementation of the function, only its header, and its docstring description.

We could probably come up with more test cases, but we'll stop here.  Writing test cases requires effort, and that effort has diminishing returns.  That is, after a certain point, more test cases are less and less likely to uncover new faults.  It is often more of a priority to make sure one tests all of the rare cases than to exhaustively test the more common cases since the rare cases are more likely to require special cases in the code, and special cases in the code are more likely to harbour faults.  The goal of test case generation is not to make enough of them to **guarantee** that the software is bug-free, but to do just enough testing that the probability of a bug remaining is extremely low.  Believe it or not, it requires vastly much more work to provide a guarantee or proof of correctness than it does to provide a very low probability!

#### Test Case Generation: White-Box Testing

When we generate test cases for the code we are testing, we call this *white-box testing*.  It is a metaphor similar to black-box testing, but in this case we imagine that the code is in a transparent box, and we can see everything inside.

In white-box testing we examine the code and try to think about all the different paths of execution through the code such as true and false branches of an if-statement, or loops that could execute different numbers of times.  Then we identify test cases that cause the execution of each of those paths at least once. If our code contains an if-statement, then we should write at least one test case that causes the if-statement's condition to be true, and one test case that causes it to be false.  If we have a loop in our code, we should write a test case that causes the loop to execute zero times, another that causes it to execute one time, one that causes it to execute many times, and one that causes it to execute the maximum number of times (if applicable).

Let's write some test cases for the `is_divisible_by_7` function from the previous section.  Here's the code again, followed by the test cases:

In [ ]:
def is_divisible_by_7(numbers):
	"""
	This function returns true if the list numbers 
	contains a number that is divisible by 7, 
	and returns false otherwise.
	numbers: list of numbers to check
	return: if list contains a number divisible by 7
	"""
	for i in numbers:
		if i % 7 == 0:
			return True
			
	return False

| Input | Expected output | Why this test case |
|---|---|---|
| `[]` | `False` | Cause the for-loop to be executed 0 times. |
| `[7]` | `True` | Cause the for-loop to be executed one time; cause the if-statement to be true. |
| `[1,2,7,3,5]` | `True` | Cause the for-loop to be executed many times; cause the if-statement to be true and false (on different loop iterations). |
| `[1,2,4,3,7]` | `True` | Cause the for-loop to be executed the maximum number of times (for the given input); cause the if-statement to be true and false (on different loop iterations). |

Notice how some test cases cover multiple testing criteria (in this case, the number of loop iterations and whether the if-statement condition is true or false)!  This is completely fine, and is encouraged, because it amounts to less work.

Also notice that all of the test cases we identified using the white-box method were also identified using the black-box method in terms of just the inputs and expected outputs.  It is well-known that white-box and black-box testing are complementary methods.  One will often identify the same test cases using either method, however, sometimes one method will helps us discover good tests that the other method does not.  There is no one right test case generation method to use; we can use one, or the other, or both.  In any case, the goal is to generate a good set of tests that is highly likely to find all of the faults that might be hiding in the code.

#### Implementing Tests

Once we have a suitable set of test cases, we have to write (i.e. *implement*) code that actually runs the tests so that we can see if the tests detect any faults.  This takes the form of a program that contains the implementation of the test cases.  Such a program is called a *test driver*.

Recalling that each test case lists an input and an expected output, we implement each test case by invoking the code to be tested with the listed input, and checking whether the received output is the expected output.  A fault is detected when a test case does not produce its expected output.  If a test case implementation detects a fault, the fault must be reported by printing a message to the console.  If a test case implementation does not detect a fault, it should output nothing.  In this way, **a test driver only reports on failed test cases.**  Results of successful test cases are not reported --- this makes it easy to see if any faults were detected after running the test driver.  If you see output, there's a problem!

##### Example Test Case Implementations

Implementation of test cases proceeds in the same way regardless of which method (white-box or black-box) was used to identify the test cases.  Here we will demonstrate the implementations of some test cases that we identified with white-box testing earlier for the `is_divisible_by_7` function.  Each test case is shown, and then followed by its implementation.

| Input | Expected output | Why this test case |
|---|---|---|
| `[]` | `False` | Cause the for-loop to be executed 0 times. |

In [ ]:
# call with empty list argument
result = is_divisible_by_7([])
# expected output: False
if result == True:
	print('Error: returned True when given empty list.')
	print('(no items divisible by 7)')

| Input | Expected output | Why this test case |
|---|---|---|
| `[7]` | `True` | Cause the for-loop to be executed one time; cause the if-statement to be true. |

In [ ]:
# call with single-item list containing one element divisible by 7
result = is_divisible_by_7([7])
# expected output: True
if result == False:
	print('Error: returned False when given [7] (divisible by 7)')

| Input | Expected output | Why this test case |
|---|---|---|
| `[1,2,7,3,5]` | `True` | Cause the for-loop to be executed many times; cause the if-statement to be true and false (on different loop iterations). |

In [ ]:
# call with many-item list containing one element divisible by 7
result = is_divisible_by_7([1,2,7,3,5])
# expected output: True
if result == False:
	print('Error: returned False when given [1,2,7,3,5]')
	print('(3rd item divisible by 7)')

Notice that when faults are reported, we always indicate what the fault was and why it was wrong.  Also notice that nothing is reported when faults are not detected.  The only exception is that we might print a "test complete" message when the test driver is concluded so that we can be certain that the test driver ran to completion and reported no faults.

### Debugging

Once we have identified a fault, we have to correct it.  We shall briefly discuss three *debugging* strategies here.  These strategies can be used regardless of how the fault was detected, whether it was through the normal course of use of the program, or through a more formal testing process like we have described in the previous section.

#### Debugging by Inspection

Sometimes the cause of a fault is obvious once the existence of the fault is detected.  In such cases, once can just inspect the code and make the necessary adjustments to repair the fault.

Sometimes if it is obvious where the fault is, but not **why** the fault occurred, it is useful to output to the console (using `print`) the values of variables/data that are used in the code causing the fault.  All too often a fault occurs at one line of code because some data was computed or stored incorrectly at an earlier point and it is necessary to trace the cause of the fault back to that point by inspecting the data along the way.

#### Debugging by Hand-Tracing Code

If inspection of the code and printing out data relevant to the fault does not help you fix it, then we can resort to more formal debugging techniques.  For small pieces of code, hand-tracing the execution of the code will usually uncover the cause of a fault.

In hand-tracing we simulate the execution of a program or function on paper.  We manually step through the code one line at a time and record the value of each program variable after every line of code, essentially executing the program "on paper".  This helps us identify the exact point in the execution at which the fault occurs and incorrect data is generated, and usually gives us insight into why the fault occurred and how to fix it.

It is very hard to demonstrate this process in a reading, but you will see a demonstration during class time.

#### Integrated Debuggers

For programs/functions that are larger and/or manipulate a very large amount of data, hand-tracing can become impractical.  In such case, we can turn to an *integrated debugger* for help.  An integrated debugger is a feature of a code editor (like Colab or JupyterLab) which allows you to have **the computer** step through a program one line at a time.  The computer still performs all of the execution of the program as normal, but you get to watch it happen one line at a time.  Again, this is very difficult to demonstrate in a reading, so we'll do a demonstration during class.  For now we will briefly describe three main features of integrated debuggers that we will demonstrate in class:

##### Stepping

Integrated debuggers display the program's code, the current value of all variables defined at that point in time, as well as which line of code is about to be executed next.  At your own speed you can repeatedly tell the debugger to execute the next single line of code.  You can also choose to step inside of function calls, or to execute an entire function call without stopping to look inside.

##### Inspection

Inspection allows you to dig deeper into the data associated with a particular variable.  For example, the variable inspection window allows you to look inside sequences and see, for example, what data items are stored inside of them.

##### Breakpoints

*Breakpoints* are useful for larger programs where it is impractical to step through every single line of the program.  If you tag a line of code as a breakpoint, then normal program execution will pause when execution reaches that line, and allow you to then inspect variables and/or begin stepping a line at a time.  You can also tell a debugger to resume uninterrupted execution of a program until the next breakpoint is encountered.  This allows you to quickly run parts of a program you aren't interested in (because you know the fault occurs elsewhere).

### Summary

Of course, testing and debugging doesn't end when the faults are fixed.  Code that has been modified to fix faults should be tested again to make sure no new faults were introduced in the course of fixing the previously detected faults. (You'd be surprised how often this happens!)  This makes testing and debugging an iterative process (illustrated in Figure ) and is also why investing the time to write a good test driver will save you time in the long run, since, if done right, re-testing is a simple matter of re-running the existing test driver.

One final note:  make sure you come to class to see the demonstrations of hand-tracing and integrated debuggers!  These are highly interactive processes that we cannot easily show using text and static pictures.

In [ ]:
show("flow_test_debug")

*Figure: The testing and debugging process.*

---

### The Beginning of the End

Welcome to the last chapter of CS1910!  You made it!

In the last few chapters, we have extended our skills and techniques for translation of problems into automated problem solving through programming. What was originally a small set of beakers and pipettes is now a robust tool bench that can solve a wide variety of problems.

As you begin to write larger and larger programs, you will encounter increasingly sticky problems that you need to solve as a programmer.  As you have probably discovered by now, the creation of any good piece of code involves a lot of *iterative* improvements, which is represented by the arrows that lead backwards from the current state to the previous state.  It is seldom that we solve a problem perfectly on the first try --- instead it often takes many refinements, changes, and sometimes a complete redesign of a program in order to get something that is fit for purpose.

Recall our double diamond process, and how it provides opportunities for feedback loops, where you can go back and refine any previous point, going all the way back to your initial understanding of the problem. In this final chapter of CS1910 we will explore a few things to think about when you are going back through each of the stages.

In [ ]:
show("double_diamond")

*Figure: The double diamond model of problem solving in programming.*

### Iterative Design

When we design our code, there are a number of different points where design decisions that we make will influence the quality of our code.

#### Cohesion

A big part of reuse is identifying how to have high *cohesion* in your code.  We mentioned cohesion in Chapter~ in reference to creating functions, and at its simplest it is described as ensuring that each function has one and only one job to do.

By keeping functions small and cohesive the code is more readable, and thus more maintainable over time.

#### Coupling

Related to cohesion is the idea of pieces of code being *coupled* together.  When we create one function called `sum_snowfall` and it invokes another called `get_snowfall`, we inexorably link the two together.  Any time someone wants to incorporate the `sum_snowfall` function into their own program they **must** take the other function with it.

While this seems like a small thing, imagine hundreds of functions, in dozens of programs that create a broad system of systems.  The reliance on other code becomes just a fact of programming at that point.  As your programs get bigger, you will become used to the questions about what your code relies on, and what other code relies on you.

#### Reuse

In this course, most of our programs have been quite small.  As programs get bigger, and do more sophisticated things, there will be opportunities for your code to be used in many different places.  If you have an isolated program that is only being called in one place, it has a very different design from one that is likely to be reused over and over again to solve a problem.

For example, if you were writing something that parses a bunch of data about your monthly budget, this is a very different problem to an organization managing budgets for multiple departments throughout a company.  In the latter, you as a programmer will need to be aware of a more wide range of incoming data, an understanding of what other programs are expecting as output, you will need to write much more efficient code that is called more often, and you will need to make absolutely sure that the values are correctly given the variety of data that is likely to be encountered.

We will discuss techniques for reuse of code more in CS1920.  However, you will find that as you refine your code to make it more cohesive and with as light coupling as possible, you have a solution that can be reused in many different places, making the next problem you solve easier.

### Iterative Implementation

Implementation is always an iterative process.  Often, you will write many dozens of lines of code only to throw them away.  This is not a failure, just part of the joy of programming.  Here are a few things that you should look out for as you go on into your programming career as common problems in your own code or in code you need to re-engineer.

#### Repetitive Code

There will be times that you find that you have written the same code to do the same thing within your program.  When that has happened, it is generally a sign that you can *refactor* your code and create a function that can represent that code.

In order to do this, you will need to step back and update your design.  You will need to understand what information goes into your function, and what comes out of it.  Further, you may need several functions.  When this happens, it is useful to take a step back, and actively design your function before writing code, just as when you started your program.

#### Bad Variable Names

Variable names can haunt a programmer later in a development cycle.  There is nothing worse than encountering something like the following in *your program*, let alone code from someone else!

> *A fragment for illustration — it is not complete enough to run.*
```python
def calculate_snowfall(a, b):
    #variables for later
    d1
    d2
    d3
    v1
    v2
    v3
```

Usually, in modern programming, we find less and less of this kind of code because we train programmers better and we have learned from 50 years of programming not to do this type of thing (Well ... in theory.).  For those of you who go on into software as a career, early positions often involve maintaining *legacy* code.  That is code that has been around for a long time and is being maintained to support some kind of functionality that is still needed, and in there you will no doubt find something like the above in the language flavour of choice from the 1970s.

When you do this, you will need to spend some time unpicking what the variables mean by printing off their values, hand-tracing code, and reading the limited comments.  As you do this, do yourself a favour --- use the powerful development tools of the modern age to update those variables to have names that are meaningful for the next poor programmer who has to maintain it.

Even better --- in your own code, write it so that you understand what your code does down the road (Well ... in theory.).

#### Not Using Comments

While variable and function names can carry some meaningful information, it is essential that you provide commenting throughout your code to improve its maintainability.

We have tried to train you from the beginning that you should be commenting your code with meaningful information that describes what the code *does* not what the code *is*.  However, when in the flow of programming, when you can see the code like a character from the Matrix (An important film reference from before you were born.  Ask your parents.), it is very easy to not put in commenting.  You just go with that flow, and before you know it you have 3000 lines of undocumented functions you will never remember how it works.

You can do that.  You can be in the flow, and enjoy the creative act of programming.  However, in those cases, one of the iterative parts of coding is going back and documenting what you have done.

#### Not Backing Up Code

Hopefully, all of you have gotten into a habit of backing up your essays and reports for university.  If you do, it is probably because you lost some piece of work in the past.

Just like any other piece of work, it is very important to have backups of your code.  Due to the iterative nature of programming, most developers embrace the common wisdom of having several backups at various important stages of development.  There are three key times to take a backup:

- When you have a major piece of functionality working.
- When you are about to refactor your code to make it "better".
- When you are about to change your code because you have a "great idea"™.

Of course much of the above list seems a bit silly, but the general rule is that if you are ever going to make a major change to your code, take a backup first.  It is recommended to have 3-5 backups so that you can revert back to some point earlier where you knew you had things working.

In future CS courses that you take you will explore systems that help us with these backups, such as `git` or `subversion`.

#### Not Having Fun

Programming can be hard work, but as a creative act it can be immensely fun.  In this course we have done lots of little problems, but if you really want to experience the joy that can come from programming, choose a problem that is close to your heart and try to solve it!

### What about understanding and evaluation?

There are many iterative elements to these stages as well.  In CS1920 we will start exploring some different types of requirements and evaluation that you can do on your code.  It will involve moving up several layers of abstraction again, and looking at the domain of program you are solving.

### The End of the Beginning

Thank you all for joining us for CS1910.  Hopefully, you have learned that the world is full of interesting problems, and that there are a key set of skills that computational thinking brings to the table.

After you leave this class you may never touch computer science or programming again.  When you encounter a tough problem, remember to decompose it into parts, abstract away details that you do not need, try to understand the logic of the cases where you can succeed and where you cannot, and if possible come up with an algorithm to solve it.